In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


In [3]:

import pandas as pd
import openai

In [ ]:
df_items = pd.read_json("../../data/meta_Digital_Music_2022_2023_with_category_rating_100_sample_1000.jsonl", lines=True)

In [ ]:
df_items.head()

In [ ]:
list(df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

### Preprocess title and features 

In [ ]:
def preprocess_description(row):
    return f"{row["title"]} {''.json(row["features"])}"

In [ ]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [ ]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [ ]:
df_items.head()

In [ ]:
# Get sample
list(df_items["description"].items())[0]

##### Sample 50 items from dataset

In [ ]:
df_sample = df_items.sample(50, random_state=42)

In [ ]:
len(df_sample)

In [ ]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [ ]:
data_to_embed

#### Define the embedding function

In [ ]:
response = openai.embeddings.create(
    input="Random Text",
    model="text-embedding-3-small"
)

In [ ]:
len(response.data[0].embedding)

In [7]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [ ]:
get_embedding("Hi")

#### Crete Qdrant collection

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

## Embed Data

##### Test

In [ ]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "text-embedding-3-small"
    }
)

### Amazon data

In [ ]:
pointstructs = []
for i, data in enumerate( ):
    embedding = get_embedding(data["description"])
     
    pointstruct.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [ ]:
pointstruct
len(pointstruct)

### Write embedded data to Qudrant 

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstruct
)

#### Define a function for data retrieval

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name = "Amazon-items-collection-00",
        query = query_embedding,
        limit = k
    )

    return results

### test retrieval

In [ ]:
retrieve_data("What kind of charging cards do you offer?", k=10).points()